# Agent Engine

## Step 1 - Preparando o projeto

### Instalar as libs

Caso não tenha o uv, instalar. Para instalação seguir documentação de [instalação uv](https://docs.astral.sh/uv/getting-started/installation/).

In [ ]:
!uv add "google-cloud-aiplatform[agent-engines]>=1.118.0"

### Criar .env

O .env deve conter as variáveis da célula abaixo. Onde:  
- `GOOGLE_CLOUD_PROJECT` representa o projeto onde o reasoning id será criado;  

- `GOOGLE_CLOUD_LOCATION` representa a região onde o reasoning id será criado;  

- `GOOGLE_GENAI_USE_VERTEXAI` sempre definir como True;  
    
- `GOOGLE_CLOUD_BUCKET` representa o bucket onde será persistido as configurações do reasoning engine. Para criar o reasoning engine é necessário ter um bucket. Para mais informações siga a documentação de [configuração do agent engine](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/set-up).


## Step 2 - Autenticação

Faça login do Gcloud SDK.

In [ ]:
!gcloud auth login
!gcloud auth application-default login

Defina o projeto.

In [ ]:
!gcloud config set project <project_id>
!gcloud auth application-default set-quota-project <project_id>

## Step 3 - Executar a criação do reasoning id

Defina o display name. Como recomendação defina um nome único.

In [ ]:
display_name = "meu-agent-engine"

Já está tudo pronto, execute a função abaixo.

In [ ]:
import os
from dotenv import load_dotenv

import vertexai
from vertexai import agent_engines
from google.adk.sessions import VertexAiSessionService

load_dotenv(dotenv_path=".env")

vertexai.init(
    project=os.getenv("GOOGLE_CLOUD_PROJECT"),
    location=os.getenv("GOOGLE_CLOUD_LOCATION"),
    staging_bucket=os.getenv("GOOGLE_CLOUD_BUCKET")
)

def get_agent_engine_name(display_name: str) -> str:
    agent_engine_data = agent_engines.list(filter=f'display_name="{display_name}"')
    agent_engines_list = list(agent_engine_data)
    exist_engine = True if len(agent_engines_list) > 0 else False
    if exist_engine:
        engine = agent_engines_list[0]
        resource_name = engine.resource_name
        return resource_name
    return None

def create_agent_engine(display_name: str) -> str:
    engine = agent_engines.create(display_name=display_name)
    resource_name = engine.resource_name
    return resource_name

def set_agent_engine_uri(display_name: str) -> str:
    resource_name = get_agent_engine_name(display_name=display_name)
    if not resource_name:
        resource_name = create_agent_engine(display_name=display_name)
    agent_engine_uri = f"agentengine://{resource_name}"
    return agent_engine_uri

uri = set_agent_engine_uri(display_name=display_name)
print(uri)

## Área perigosa

Defina o display_name.

In [ ]:
display_name = "meu-agent-engine"

Para deletar um reasoning engine, utilize a função abaixo.

In [ ]:
def delete_engine(display_name: str) -> None:
    resource_name = get_agent_engine_name(display_name=display_name)
    agent_engines.delete(resource_name)

delete_engine(display_name=display_name)